Quick sanity check: how are different models currently embedding certain users? \
(big matrix only, to prevent leakage)

In [3]:
!pwd

/dcs/23/u5567816/deep-linear-bandits/notebooks


In [4]:
# Load in big matrix

import deep_linear_bandits.data as dlb_data

DLB_DIR = "/dcs/23/u5567816/deep-linear-bandits/"

big_train, big_val = dlb_data.preprocess_krbig_interactions(data_dir=DLB_DIR+"kuairec/data/")

In [5]:
# Get item popularities in big matrix

import numpy as np
import pandas as pd

item_popularity = dlb_data.compute_item_popularity(
    interactions=pd.concat([big_train, big_val]).reset_index(drop=True),
    watch_threshold=2.0
)

In [6]:
# Show items sorted by global popularity

pd.DataFrame({
    'global_popularity': item_popularity
}).sort_values(by=['global_popularity'], ascending=False)

,global_popularity
314,2559.0
7383,2332.0
8298,2253.0
8366,1967.0
3400,1829.0
...,...
10601,0.0
6216,0.0
6218,0.0
10669,0.0


In [ ]:
# Get all user & item embeddings for a specific MODEL

import deep_linear_bandits.two_tower as dlb_tt
import torch
import pickle

pd.set_option('display.min_rows', 25)

NUM_USERS = 7176
NUM_ITEMS = 10728

MODEL = "fsw-t0.10"

torch.set_float32_matmul_precision('high')
device = (
    torch.accelerator.current_accelerator()
    if torch.accelerator.is_available()
    else torch.device('cpu')
)

with open(DLB_DIR + f"tt-models/{MODEL}/model_args.pkl", "rb") as f:
    model_args = pickle.load(f)
model = dlb_tt.TwoTower(**model_args).to(device)
model.load_state_dict(
    torch.load(
        DLB_DIR + f"tt-models/{MODEL}/model.pt",
        map_location=device,
        weights_only=True
    )
)
model.eval()

with torch.inference_mode():
    """
    Retrieve embeddings for all users & items in the big matrix
    """
    user_ids = torch.arange(0, NUM_USERS, device=device, dtype=torch.long)
    (user_cat_feats, _), user_numeric_feats = dlb_data.preprocess_user_features(data_dir=DLB_DIR+"kuairec/data/")
    user_cat_feats = torch.tensor(user_cat_feats, dtype=torch.long, device=device)
    user_numeric_feats = torch.tensor(user_numeric_feats, dtype=torch.float32, device=device)

    user_embeddings = model.user_tower(user_ids, user_cat_feats, user_numeric_feats).cpu().numpy()

    item_ids = torch.arange(0, NUM_ITEMS, device=device, dtype=torch.long)
    item_categories = dlb_data.preprocess_item_categories(data_dir=DLB_DIR+"kuairec/data/").to(device)

    item_embeddings = model.item_tower(item_ids, item_categories).cpu().numpy()

In [57]:
# What's a specific user U's top items in the big matrix?
# (from training, avoid leakage of val)

U = 1376 # (recall users are between 0 and 7175)

U_wrs = big_train[big_train.user_id == U].sort_values(by='watch_ratio', ascending=False)
U_wrs = U_wrs.rename(columns = {'video_id': 'item_id'})

# What's the model currently got as the top items for that user?
# user_embeddings: (7176, D); for one user it's (D,)
# item_embeddings: (10728, D)
item_scores = item_embeddings @ user_embeddings[U, :]
top_items = np.argsort(item_scores)[::-1]
top_items_scores = item_scores[top_items]

df = pd.DataFrame({
    'item_id': top_items,
    'model_similarity': top_items_scores
})

# Top items by model similarity, and the actual watch_ratio (+ item popularity)
top_model = df.merge(right=U_wrs, on='item_id')
top_model['global_popularity'] = item_popularity[top_model.item_id.values]
top_model

,item_id,model_similarity,user_id,watch_ratio,global_popularity
0,5464,0.907418,1376,1.774618,1802.0
1,314,0.896868,1376,2.399087,2559.0
2,8298,0.889964,1376,2.102752,2253.0
3,4040,0.868530,1376,2.753567,1481.0
4,2130,0.847776,1376,2.723297,1417.0
5,154,0.838119,1376,2.291443,1322.0
6,4123,0.832784,1376,1.582484,1455.0
7,2123,0.810732,1376,1.610614,1582.0
8,10017,0.805531,1376,2.021087,903.0
9,5434,0.801476,1376,2.527463,1193.0


In [58]:
# Top items by their actual watch_ratio; index will be where the model is actually placing them
top_model.sort_values(by=['watch_ratio'], ascending=False)

,item_id,model_similarity,user_id,watch_ratio,global_popularity
278,10184,0.030520,1376,7.392721,430.0
24,6463,0.716392,1376,6.188506,464.0
32,3401,0.669033,1376,4.419501,617.0
1332,867,-0.246944,1376,3.951360,115.0
995,5738,-0.155783,1376,3.612493,97.0
506,2110,-0.053250,1376,3.318202,16.0
1388,7296,-0.266939,1376,3.276265,51.0
1177,4261,-0.199041,1376,3.173942,162.0
769,4425,-0.114262,1376,3.097894,140.0
10,1894,0.795783,1376,2.986958,658.0


In [59]:
# Top items by their global popularity, index is where the model is placing them
top_model.sort_values(by=['global_popularity'], ascending=False)

,item_id,model_similarity,user_id,watch_ratio,global_popularity
1,314,0.896868,1376,2.399087,2559.0
2,8298,0.889964,1376,2.102752,2253.0
18,3400,0.760623,1376,2.256117,1829.0
0,5464,0.907418,1376,1.774618,1802.0
7,2123,0.810732,1376,1.610614,1582.0
17,2894,0.767376,1376,1.399328,1579.0
3,4040,0.868530,1376,2.753567,1481.0
6,4123,0.832784,1376,1.582484,1455.0
40,1709,0.596160,1376,2.021032,1434.0
4,2130,0.847776,1376,2.723297,1417.0
